# RAG sobre HTTPX

Construir um sistesma de recuperação semântica sobre a documentação do projeto HTTPX.

Fluxo:

repositório → arquivos Markdown → chunks + metadados → embeddings
→ busca por similaridade → trechos relevantes + fontes

A geração de resposta com Gemma 3 será utilizada como extensão opcional.

# Instalação e importação de bibliotecas

Instalação do sentence-transformers que será usado para os embeddings. E transformers será usado para carregar o Gemma.



In [2]:
!pip install -q -U sentence-transformers transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.7 MB/s eta 0:00:00


In [45]:
import os
import re
import glob
import textwrap
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sentence_transformers import SentenceTransformer


# Conferindo Colab

Conferindo se o colab esta usando a GPU

In [4]:
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA disponível: True
GPU: Tesla T4


#Clonar o HTTPX

Aqui será realizado a clonagem do arquivo HTTPX

In [5]:
!git clone https://github.com/encode/httpx.git

Cloning into 'httpx'...
remote: Enumerating objects: 10983, done.
remote: Total 10983 (delta 0), reused 0 (delta 0), pack-reused 10983 (from 1)
Receiving objects: 100% (10983/10983), 8.41 MiB | 17.62 MiB/s, done.
Resolving deltas: 100% (8278/8278), done.


In [6]:
%cd httpx

/content/httpx


In [7]:
!git checkout b5addb64f0161ff6bfe94c124ef76f6a1fba5254


Note: switching to 'b5addb64f0161ff6bfe94c124ef76f6a1fba5254'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at b5addb6 Adapt test_response_decode_text_using_autodetect for chardet 6.0 (#3773)


#Verificação dos arquivos

Aqui foi feito a verificação de todos arquivos Markdown dentro da pasta httpx/docs

In [8]:
from pathlib import Path

docs_dir = Path("docs")
md_files = sorted(docs_dir.rglob("*.md"))
print(f"Arquivos encontrados: {len(md_files)}")
assert len(md_files) == 23, "Contagem diferente do esperado — confira a pasta e o commit."
for f in md_files:
    print(" -", f)

Arquivos encontrados: 23
 - docs/advanced/authentication.md
 - docs/advanced/clients.md
 - docs/advanced/event-hooks.md
 - docs/advanced/extensions.md
 - docs/advanced/proxies.md
 - docs/advanced/resource-limits.md
 - docs/advanced/ssl.md
 - docs/advanced/text-encodings.md
 - docs/advanced/timeouts.md
 - docs/advanced/transports.md
 - docs/api.md
 - docs/async.md
 - docs/code_of_conduct.md
 - docs/compatibility.md
 - docs/contributing.md
 - docs/environment_variables.md
 - docs/exceptions.md
 - docs/http2.md
 - docs/index.md
 - docs/logging.md
 - docs/quickstart.md
 - docs/third_party_packages.md
 - docs/troubleshooting.md


#Lendo os documentos

Carregando o conteudo

In [9]:
documents = []

for path in md_files:
    text = path.read_text(encoding="utf-8", errors="ignore")

    documents.append({
        "file": str(path),
        "text": text
    })

print(f"Documentos carregados: {len(documents)}")

Documentos carregados: 23


In [10]:
print(documents[0]["file"])
print(documents[0]["text"][:2000])

docs/advanced/authentication.md
Authentication can either be included on a per-request basis...

```pycon
>>> auth = httpx.BasicAuth(username="username", password="secret")
>>> client = httpx.Client()
>>> response = client.get("https://www.example.com/", auth=auth)
```

Or configured on the client instance, ensuring that all outgoing requests will include authentication credentials...

```pycon
>>> auth = httpx.BasicAuth(username="username", password="secret")
>>> client = httpx.Client(auth=auth)
>>> response = client.get("https://www.example.com/")
```

## Basic authentication

HTTP basic authentication is an unencrypted authentication scheme that uses a simple encoding of the username and password in the request `Authorization` header. Since it is unencrypted it should typically only be used over `https`, although this is not strictly enforced.

```pycon
>>> auth = httpx.BasicAuth(username="finley", password="secret")
>>> client = httpx.Client(auth=auth)
>>> response = client.get("ht

# Realização dos Chunks

In [11]:
CHUNK_SIZE = 85
OVERLAP = 15

def detectar_secao(texto):
    linhas = texto.splitlines()

    for linha in linhas:
        linha = linha.strip()

        if linha.startswith("#"):
            return linha.lstrip("#").strip()

    return "Sem título identificado"

In [12]:
def dividir_em_chunks(texto, chunk_size=85, overlap=15):
    palavras = texto.split()

    if not palavras:
        return []

    chunks = []
    inicio = 0

    while inicio < len(palavras):
        fim = inicio + chunk_size
        chunk = " ".join(palavras[inicio:fim]).strip()

        if chunk:
            chunks.append(chunk)

        if fim >= len(palavras):
            break

        inicio = fim - overlap

    return chunks

In [13]:
chunks = []
metadata = []

chunk_id = 0

for doc in documents:
    file_path = doc["file"]
    text = doc["text"]
    section = detectar_secao(text)

    doc_chunks = dividir_em_chunks(
        text,
        chunk_size=CHUNK_SIZE,
        overlap=OVERLAP
    )

    for chunk in doc_chunks:
        chunks.append(chunk)

        metadata.append({
            "chunk_id": chunk_id,
            "file": file_path,
            "section": section
        })

        chunk_id += 1

print("Total de chunks:", len(chunks))
print("Total de metadados:", len(metadata))
assert len(chunks) == len(metadata)

Total de chunks: 234
Total de metadados: 234


In [14]:
for i in range(min(10, len(chunks))):
    print("=" * 80)
    print("CHUNK:", i)
    print("ARQUIVO:", metadata[i]["file"])
    print("SEÇÃO:", metadata[i]["section"])
    print()
    print(textwrap.fill(chunks[i], width=100))

CHUNK: 0
ARQUIVO: docs/advanced/authentication.md
SEÇÃO: Basic authentication

Authentication can either be included on a per-request basis... ```pycon >>> auth =
httpx.BasicAuth(username="username", password="secret") >>> client = httpx.Client() >>> response =
client.get("https://www.example.com/", auth=auth) ``` Or configured on the client instance, ensuring
that all outgoing requests will include authentication credentials... ```pycon >>> auth =
httpx.BasicAuth(username="username", password="secret") >>> client = httpx.Client(auth=auth) >>>
response = client.get("https://www.example.com/") ``` ## Basic authentication HTTP basic
authentication is an unencrypted authentication scheme that uses a simple encoding of the username
and password in the request `Authorization` header. Since it is unencrypted
CHUNK: 1
ARQUIVO: docs/advanced/authentication.md
SEÇÃO: Basic authentication

encoding of the username and password in the request `Authorization` header. Since it is unencrypted
it sho

# Embemdding

Aqui será a parte em que vamos vetorizar estes chunks e colocalos num banco veotrial

In [15]:
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedder = SentenceTransformer(EMBEDDING_MODEL)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer(
    EMBEDDING_MODEL,
    device=device
)

print("Modelo:", EMBEDDING_MODEL)
print("Dispositivo:", device)

#Geração dos embeddings dos chunks feitos

embeddings = embedder.encode(
    chunks,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(embeddings)

print("Formato dos embeddings:", embeddings.shape)
normalize_embeddings=True

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Dispositivo: cuda


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Formato dos embeddings: (234, 384)


# Busca por similaridade

In [17]:
def buscar(consulta, top_k=5):
    if not consulta or not consulta.strip():
        raise ValueError("A pergunta não pode estar vazia.")

    if not isinstance(top_k, int) or top_k < 1 or top_k > 5:
        raise ValueError("top_k deve ser um inteiro entre 1 e 5.")

    if len(chunks) == 0:
        raise ValueError("O corpus não possui documentos indexados.")

    query_embedding = embedder.encode(
        [consulta],
        normalize_embeddings=True
    )[0]

    scores = embeddings @ query_embedding

    indices = np.argsort(scores)[::-1][:top_k]

    resultados = []

    for rank, idx in enumerate(indices, start=1):
        resultados.append({
            "rank": rank,
            "score": float(scores[idx]),
            "text": chunks[idx],
            "file": metadata[idx]["file"],
            "section": metadata[idx]["section"],
            "chunk_id": metadata[idx]["chunk_id"]
        })

    return resultados

In [18]:
#Mostrar o resultado visualmente melhor

def mostrar_resultados(resultados):
    for resultado in resultados:
        print("=" * 100)
        print(f"RANK: {resultado['rank']}")
        print(f"SCORE: {resultado['score']:.4f}")
        print(f"ARQUIVO: {resultado['file']}")
        print(f"SEÇÃO: {resultado['section']}")
        print(f"CHUNK ID: {resultado['chunk_id']}")
        print("-" * 100)
        print(textwrap.fill(resultado["text"], width=100))
        print()

# Teste de perguntas

Realização de teste com 3 perguntas em português

In [19]:
pergunta = "Como posso configurar um timeout no HTTPX?"

resultados = buscar(pergunta, top_k=5)

mostrar_resultados(resultados)

RANK: 1
SCORE: 0.8133
ARQUIVO: docs/advanced/timeouts.md
SEÇÃO: Setting and disabling timeouts
CHUNK ID: 77
----------------------------------------------------------------------------------------------------
HTTPX is careful to enforce timeouts everywhere by default. The default behavior is to raise a
`TimeoutException` after 5 seconds of network inactivity. ## Setting and disabling timeouts You can
set timeouts for an individual request: ```python # Using the top-level API:
httpx.get('http://example.com/api/v1/example', timeout=10.0) # Using a client instance: with
httpx.Client() as client: client.get("http://example.com/api/v1/example", timeout=10.0) ``` Or
disable timeouts for an individual request: ```python # Using the top-level API:
httpx.get('http://example.com/api/v1/example', timeout=None) # Using a client instance: with
httpx.Client() as client: client.get("http://example.com/api/v1/example", timeout=None) ``` ##
Setting

RANK: 2
SCORE: 0.7715
ARQUIVO: docs/advanced/timeouts

In [20]:
pergunta_2 = "Como o HTTPX gerencia conexões?"

resultados_2 = buscar(pergunta_2, top_k=5)

mostrar_resultados(resultados_2)

RANK: 1
SCORE: 0.6741
ARQUIVO: docs/advanced/proxies.md
SEÇÃO: HTTP Proxies
CHUNK ID: 55
----------------------------------------------------------------------------------------------------
HTTPX supports setting up [HTTP
proxies](https://en.wikipedia.org/wiki/Proxy_server#Web_proxy_servers) via the `proxy` parameter to
be passed on client initialization or top-level API functions like `httpx.get(..., proxy=...)`. <div
align="center"> <img
src="https://upload.wikimedia.org/wikipedia/commons/thumb/2/27/Open_proxy_h2g2bob.svg/480px-
Open_proxy_h2g2bob.svg.png"/> <figcaption><em>Diagram of how a proxy works (source: Wikipedia). The
left hand side "Internet" blob may be your HTTPX client requesting <code>example.com</code> through
a proxy.</em></figcaption> </div> ## HTTP Proxies To route all traffic (HTTP and HTTPS) to a proxy
located at `http://localhost:8030`, pass the proxy URL to the client... ```python with
httpx.Client(proxy="http://localhost:8030") as client: ... ``` For more advan

In [21]:
pergunta_3 = "Passei na fabrica de software?"

resultados_3 = buscar(pergunta_3, top_k=5)

mostrar_resultados(resultados_3)

RANK: 1
SCORE: 0.2896
ARQUIVO: docs/contributing.md
SEÇÃO: Contributing
CHUNK ID: 162
----------------------------------------------------------------------------------------------------
lint job'> </p> This job failing means there is either a code formatting issue or type-annotation
issue. You can look at the job output to figure out why it's failed or within a shell run: ```shell
$ scripts/check ``` It may be worth it to run `$ scripts/lint` to attempt auto-formatting the code
and if that job succeeds commit the changes. ### Docs Job Failed This job failing means the
documentation failed to build. This can happen for a variety of reasons like invalid markdown

RANK: 2
SCORE: 0.2851
ARQUIVO: docs/contributing.md
SEÇÃO: Contributing
CHUNK ID: 158
----------------------------------------------------------------------------------------------------
exist when using `asyncio` or `trio`, or both? ## Development To start developing HTTPX create a
**fork** of the [HTTPX repository](https://gi

In [22]:
#Aqui será um comando de busca interativa com o usuario

pergunta = input("Digite sua pergunta: ")

resultados = buscar(pergunta, top_k=5)

mostrar_resultados(resultados)

Digite sua pergunta: o que httpx
RANK: 1
SCORE: 0.6000
ARQUIVO: docs/advanced/proxies.md
SEÇÃO: HTTP Proxies
CHUNK ID: 55
----------------------------------------------------------------------------------------------------
HTTPX supports setting up [HTTP
proxies](https://en.wikipedia.org/wiki/Proxy_server#Web_proxy_servers) via the `proxy` parameter to
be passed on client initialization or top-level API functions like `httpx.get(..., proxy=...)`. <div
align="center"> <img
src="https://upload.wikimedia.org/wikipedia/commons/thumb/2/27/Open_proxy_h2g2bob.svg/480px-
Open_proxy_h2g2bob.svg.png"/> <figcaption><em>Diagram of how a proxy works (source: Wikipedia). The
left hand side "Internet" blob may be your HTTPX client requesting <code>example.com</code> through
a proxy.</em></figcaption> </div> ## HTTP Proxies To route all traffic (HTTP and HTTPS) to a proxy
located at `http://localhost:8030`, pass the proxy URL to the client... ```python with
httpx.Client(proxy="http://localhost:8030") 

# Adição GEMMA

Vamos adicoinar o GEMMA 3 1B

In [43]:
#Guardando o token no colab

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN:
    print("Token carregado com sucesso!")
else:
    print("Token não encontrado.")

Token carregado com sucesso!


In [44]:
#Carregaremos o GEMMA

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GEMMA_MODEL = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(
    GEMMA_MODEL,
    token=HF_TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    GEMMA_MODEL,
    token=HF_TOKEN,
    device_map="auto",
    torch_dtype="auto"
)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [25]:
#Realização de teste do GEMMA
#Para ter a certeza que não irá bugar na RAG

messages = [
    {
        "role": "user",
        "content": "Explique em uma frase o que é o HTTPX."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

resposta = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(resposta)

O HTTPX é um protocolo HTTP aberto e moderno que visa simplificar o desenvolvimento de aplicações web, oferecendo uma alternativa ao HTTP tradicional e facilitando a implementação de APIs de forma mais consistente e robusta.


In [26]:
#Agora adiocinado dentro da RAG

def montar_contexto(resultados):
    partes = []

    for resultado in resultados:
        partes.append(
            f"[Fonte {resultado['rank']}]\n"
            f"Arquivo: {resultado['file']}\n"
            f"Seção: {resultado['section']}\n"
            f"Trecho:\n{resultado['text']}"
        )

    return "\n\n".join(partes)

def criar_prompt(pergunta, contexto):         #Está função serve para que o GEMMA não responda com conhecimentos próprios
    return f"""
Você é um assistente que responde perguntas usando exclusivamente
os trechos de documentação fornecidos abaixo.

Regras:
1. Responda apenas com base no contexto.
2. Não invente informações.
3. Se o contexto não contiver a resposta, diga claramente:
   "Não encontrei essa informação na documentação recuperada."
4. Seja objetivo.
5. Ao final, informe quais fontes foram utilizadas.

PERGUNTA:
{pergunta}

CONTEXTO:
{contexto}
"""

In [27]:
#Função de geração
def gerar_resposta_gemma(pergunta, resultados):
    contexto = montar_contexto(resultados)
    prompt = criar_prompt(pergunta, contexto)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    resposta = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return resposta

In [28]:
#Juntando tudo para RAG ficar completa
def rag(pergunta, top_k=3):
    resultados = buscar(pergunta, top_k=top_k)

    print("=" * 100)
    print("RESULTADOS DA RECUPERAÇÃO")
    print("=" * 100)

    mostrar_resultados(resultados)

    print()
    print("=" * 100)
    print("RESPOSTA DO GEMMA")
    print("=" * 100)

    resposta = gerar_resposta_gemma(
        pergunta,
        resultados
    )

    print(resposta)

    return resultados, resposta

# Realizando o primeiro RAG

Realizando as 3 perguntass novamente com o GEMMA incluido agora

In [29]:
resultados, resposta = rag(
    "Como configurar timeout no HTTPX?",
    top_k=5
)

RESULTADOS DA RECUPERAÇÃO
RANK: 1
SCORE: 0.8292
ARQUIVO: docs/advanced/timeouts.md
SEÇÃO: Setting and disabling timeouts
CHUNK ID: 77
----------------------------------------------------------------------------------------------------
HTTPX is careful to enforce timeouts everywhere by default. The default behavior is to raise a
`TimeoutException` after 5 seconds of network inactivity. ## Setting and disabling timeouts You can
set timeouts for an individual request: ```python # Using the top-level API:
httpx.get('http://example.com/api/v1/example', timeout=10.0) # Using a client instance: with
httpx.Client() as client: client.get("http://example.com/api/v1/example", timeout=10.0) ``` Or
disable timeouts for an individual request: ```python # Using the top-level API:
httpx.get('http://example.com/api/v1/example', timeout=None) # Using a client instance: with
httpx.Client() as client: client.get("http://example.com/api/v1/example", timeout=None) ``` ##
Setting

RANK: 2
SCORE: 0.7867
ARQUI

In [30]:
resultados, resposta = rag(
    "Como o HTTPX gerencia conexões?",
    top_k=5
)

RESULTADOS DA RECUPERAÇÃO
RANK: 1
SCORE: 0.6741
ARQUIVO: docs/advanced/proxies.md
SEÇÃO: HTTP Proxies
CHUNK ID: 55
----------------------------------------------------------------------------------------------------
HTTPX supports setting up [HTTP
proxies](https://en.wikipedia.org/wiki/Proxy_server#Web_proxy_servers) via the `proxy` parameter to
be passed on client initialization or top-level API functions like `httpx.get(..., proxy=...)`. <div
align="center"> <img
src="https://upload.wikimedia.org/wikipedia/commons/thumb/2/27/Open_proxy_h2g2bob.svg/480px-
Open_proxy_h2g2bob.svg.png"/> <figcaption><em>Diagram of how a proxy works (source: Wikipedia). The
left hand side "Internet" blob may be your HTTPX client requesting <code>example.com</code> through
a proxy.</em></figcaption> </div> ## HTTP Proxies To route all traffic (HTTP and HTTPS) to a proxy
located at `http://localhost:8030`, pass the proxy URL to the client... ```python with
httpx.Client(proxy="http://localhost:8030") as clie

In [31]:
resultados, resposta = rag(
    "Passei na fabrica de software?",
    top_k=3
)

RESULTADOS DA RECUPERAÇÃO
RANK: 1
SCORE: 0.2896
ARQUIVO: docs/contributing.md
SEÇÃO: Contributing
CHUNK ID: 162
----------------------------------------------------------------------------------------------------
lint job'> </p> This job failing means there is either a code formatting issue or type-annotation
issue. You can look at the job output to figure out why it's failed or within a shell run: ```shell
$ scripts/check ``` It may be worth it to run `$ scripts/lint` to attempt auto-formatting the code
and if that job succeeds commit the changes. ### Docs Job Failed This job failing means the
documentation failed to build. This can happen for a variety of reasons like invalid markdown

RANK: 2
SCORE: 0.2851
ARQUIVO: docs/contributing.md
SEÇÃO: Contributing
CHUNK ID: 158
----------------------------------------------------------------------------------------------------
exist when using `asyncio` or `trio`, or both? ## Development To start developing HTTPX create a
**fork** of the [HT

In [32]:
#Aqui são tratamentos para caso houver pergunta vazia, tok-inválido ou  corpus vazio
try:
    buscar("")
except ValueError as e:
    print("Erro:", e)
try:
    buscar("Como funciona o HTTPX?", top_k=10)
except ValueError as e:
    print("Erro:", e)
backup_chunks = chunks.copy()

chunks.clear()

try:
    buscar("Como funciona o HTTPX?")
except ValueError as e:
    print("Erro:", e)

chunks.extend(backup_chunks)

Erro: A pergunta não pode estar vazia.
Erro: top_k deve ser um inteiro entre 1 e 5.
Erro: O corpus não possui documentos indexados.


# Refinamento

Deixando mais robusto a consulta e respota (GEMMA demostrando a fonte)

Obs: Um extra


In [33]:
def rag_com_fallback(pergunta, top_k=5):
    resultados = buscar(pergunta, top_k=top_k)

    print("=" * 100)
    print("TRECHOS RECUPERADOS")
    print("=" * 100)

    mostrar_resultados(resultados)

    try:
        resposta = gerar_resposta_gemma(
            pergunta,
            resultados
        )

        print("=" * 100)
        print("RESPOSTA DO GEMMA")
        print("=" * 100)
        print(resposta)

    except Exception as e:
        print("=" * 100)
        print("GERAÇÃO INDISPONÍVEL")
        print("=" * 100)
        print("A recuperação semântica funcionou, mas o Gemma não pôde ser executado.")
        print("Erro:", str(e))

In [34]:
def executar_rag(pergunta, top_k=5):
    resultados = buscar(pergunta, top_k=top_k)

    print("=" * 100)
    print("PERGUNTA")
    print("=" * 100)
    print(pergunta)

    print("\n")
    print("=" * 100)
    print("TOP RESULTADOS")
    print("=" * 100)

    mostrar_resultados(resultados)

    try:
        resposta = gerar_resposta_gemma(pergunta, resultados)

        print("=" * 100)
        print("RESPOSTA GERADA PELO GEMMA 3")
        print("=" * 100)
        print(resposta)

    except Exception as e:
        print("=" * 100)
        print("FALLBACK")
        print("=" * 100)
        print("Não foi possível gerar a resposta com o Gemma.")
        print(f"Motivo: {e}")

    print("\n")
    print("=" * 100)
    print("FONTES")
    print("=" * 100)

    for r in resultados:
        print(f"{r['rank']}. {r['file']} — {r['section']}")

    return resultados

In [46]:
executar_rag(
    "Como configurar timeout no HTTPX?",
    top_k=3
)

PERGUNTA
Como configurar timeout no HTTPX?


TOP RESULTADOS
RANK: 1
SCORE: 0.8292
ARQUIVO: docs/advanced/timeouts.md
SEÇÃO: Setting and disabling timeouts
CHUNK ID: 77
----------------------------------------------------------------------------------------------------
HTTPX is careful to enforce timeouts everywhere by default. The default behavior is to raise a
`TimeoutException` after 5 seconds of network inactivity. ## Setting and disabling timeouts You can
set timeouts for an individual request: ```python # Using the top-level API:
httpx.get('http://example.com/api/v1/example', timeout=10.0) # Using a client instance: with
httpx.Client() as client: client.get("http://example.com/api/v1/example", timeout=10.0) ``` Or
disable timeouts for an individual request: ```python # Using the top-level API:
httpx.get('http://example.com/api/v1/example', timeout=None) # Using a client instance: with
httpx.Client() as client: client.get("http://example.com/api/v1/example", timeout=None) ``` ##
Se

[{'rank': 1,
  'score': 0.8292186260223389,
  'text': 'HTTPX is careful to enforce timeouts everywhere by default. The default behavior is to raise a `TimeoutException` after 5 seconds of network inactivity. ## Setting and disabling timeouts You can set timeouts for an individual request: ```python # Using the top-level API: httpx.get(\'http://example.com/api/v1/example\', timeout=10.0) # Using a client instance: with httpx.Client() as client: client.get("http://example.com/api/v1/example", timeout=10.0) ``` Or disable timeouts for an individual request: ```python # Using the top-level API: httpx.get(\'http://example.com/api/v1/example\', timeout=None) # Using a client instance: with httpx.Client() as client: client.get("http://example.com/api/v1/example", timeout=None) ``` ## Setting',
  'file': 'docs/advanced/timeouts.md',
  'section': 'Setting and disabling timeouts',
  'chunk_id': 77},
 {'rank': 2,
  'score': 0.7866657376289368,
  'text': 'httpx.Client(timeout=None) # Disable all t

# Resultado Final do GEMMA
Aqui colcoamos o usuario para realizar a busca interrativa com o GEMMA

In [36]:
#recuperar os resultados da rag e da contexto para GEMMA
def montar_contexto(resultados):
    partes = []

    for r in resultados:
        partes.append(
            f"""
[FONTE {r['rank']}]
Arquivo: {r['file']}
Seção: {r['section']}
Score: {r['score']:.4f}

Trecho:
{r['text']}
"""
        )

    return "\n".join(partes)

In [38]:
#aqui será como vamos chama o GEMMA
def gerar_resposta_gemma(pergunta, resultados):

    contexto = montar_contexto(resultados)

    prompt = criar_prompt(
        pergunta,
        contexto
    )

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    novos_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    resposta = tokenizer.decode(
        novos_tokens,
        skip_special_tokens=True
    )

    return resposta.strip()

In [39]:
#essa função funciona para interação do usuario

def busca_interativa():

    print("=" * 80)
    print("        RAG HTTPX + GEMMA 3")
    print("=" * 80)
    print("Digite sua pergunta.")
    print("Digite 'sair' para encerrar.")
    print()

    while True:

        pergunta = input("Pergunta: ").strip()

        if pergunta.lower() == "sair":
            print("\nEncerrando...")
            break

        if not pergunta:
            print("⚠️ Digite uma pergunta válida.\n")
            continue

        try:

            # 1. Recuperação semântica
            resultados = buscar(
                pergunta,
                top_k=3
            )

            # 2. Mostrar os documentos recuperados
            print("\n" + "=" * 80)
            print("TOP 3 TRECHOS RECUPERADOS")
            print("=" * 80)

            for r in resultados:

                print(f"\n[{r['rank']}] Score: {r['score']:.4f}")
                print(f"Arquivo: {r['file']}")
                print(f"Seção: {r['section']}")
                print("-" * 80)
                print(textwrap.fill(r["text"], width=100))

            # 3. Gerar resposta
            print("\n" + "=" * 80)
            print("RESPOSTA DO GEMMA")
            print("=" * 80)

            resposta = gerar_resposta_gemma(
                pergunta,
                resultados
            )

            print(resposta)

            # 4. Mostrar fontes
            print("\n" + "=" * 80)
            print("FONTES UTILIZADAS")
            print("=" * 80)

            for r in resultados:
                print(
                    f"{r['rank']}. "
                    f"{r['file']} "
                    f"— {r['section']}"
                )

            print()

        except Exception as e:

            print("\n Ocorreu um erro:")
            print(str(e))
            print()

In [40]:

busca_interativa()

        RAG HTTPX + GEMMA 3
Digite sua pergunta.
Digite 'sair' para encerrar.

Pergunta: sair

Encerrando...
